# qstl_awg_tuning_fir FIR DDR Capture Demo

This notebook is for firmware built from `qstl_awg_tuning_fir`, where the DDR capture path is:

```text
readout stream -> axis_fir_decim_300to1_v1 -> axis_buffer_ddr_sample_v1 -> PL DDR
```

It contains:

- a QICK `AveragerProgram` that produces generator/AWG activity and triggers the FIR DDR path,
- an execution example that arms the DRAM IP with `soc.arm_ddr4_fir_samples()`, runs the program, and reads data with `soc.get_ddr4_fir_samples()`,
- a hardware guard so the notebook can be opened without immediately connecting to a board.

Important: the DDR sample buffer `sample_decim` is not used to make 1 MSPS here. The anti-alias downsampling is performed by `axis_fir_decim_300to1_v1`, and the DDR buffer is armed with `sample_decim=1` internally.


Authors: Jeonghyun Park (jeonghyun.park@ubc.ca or alexist@snu.ac.kr), Farbod


In [ ]:
from pathlib import Path
import sys
import numpy as np

REPO = Path(r"C:\JeonghyunPark\Workspace\QSTL_QICK")
QICK_LIB = REPO / "qick" / "qick_lib"
if str(QICK_LIB) not in sys.path:
    sys.path.insert(0, str(QICK_LIB))

from qick import AveragerProgram

FIRMWARE_DIR = REPO / "qick" / "firmware" / "projects" / "qstl_awg_tuning_fir"
BITFILE = FIRMWARE_DIR / "bitstream.bit"
HWHFILE = FIRMWARE_DIR / "bitstream.hwh"
XSAFILE = FIRMWARE_DIR / "bitstream.xsa"

print("QICK lib:", QICK_LIB)
print("Firmware dir:", FIRMWARE_DIR)
print("bit exists:", BITFILE.exists())
print("hwh exists:", HWHFILE.exists())
print("xsa exists:", XSAFILE.exists())


## Connect to hardware

Set `RUN_HARDWARE = True` only when the `qstl_awg_tuning_fir` bitstream is loaded on the board or the remote QICK server is configured to use that firmware.

The notebook uses Pyro because that is the same style as the loopback snippets used in this repository.


In [ ]:
RUN_HARDWARE = False
QICK_PROXY_ADDR = "192.168.2.99"

if RUN_HARDWARE:
    from qick.pyro import make_proxy
    soc, soccfg = make_proxy(QICK_PROXY_ADDR)
    print(soccfg)
else:
    soc = None
    soccfg = None
    print("Hardware connection disabled. Set RUN_HARDWARE=True to connect.")


## Firmware sanity check

This check verifies that the loaded HWH exposes the new sample-count DDR capture driver. The expected buffer config contains `sample_capture=True`.


In [ ]:
def require_sample_ddr4(soccfg):
    if soccfg is None:
        print("No hardware config loaded yet.")
        return None

    ddr_cfg = soccfg["ddr4_buf"]
    print("DDR4 buffer config:")
    for key in [
        "type",
        "fullpath",
        "sample_capture",
        "fir_enabled",
        "fir_fullpath",
        "fir_decimation",
        "fir_input_fs_mhz",
        "fir_output_fs_mhz",
        "fir_group_delay_input_samples",
        "s_axis_data_width",
        "m_axi_data_width",
        "samples_per_axi_word",
        "bytes_per_axi_word",
        "trigger_port",
        "trigger_bit",
        "maxlen",
    ]:
        print(f"  {key}: {ddr_cfg.get(key)}")

    if not ddr_cfg.get("sample_capture", False):
        raise RuntimeError("This firmware does not expose AxisBufferDdrSampleV1; load qstl_awg_tuning_fir.")
    if not ddr_cfg.get("fir_enabled", False):
        raise RuntimeError("This firmware does not expose axis_fir_decim_300to1_v1 in the DDR path.")
    if ddr_cfg.get("samples_per_axi_word") != 8:
        raise RuntimeError("Expected 8 32-bit samples per 256-bit AXI word.")
    return ddr_cfg

if RUN_HARDWARE:
    ddr_cfg = require_sample_ddr4(soccfg)


## Pick channels

The helper selects one normal `axis_signal_gen_v6`, one `axis_awg_tuning_v1`, and one readout channel. Override these values if your hardware wiring requires different channels.


In [ ]:
def find_first_gen(soccfg, *, gen_type=None, gen_type_key=None):
    for idx, gen in enumerate(soccfg["gens"]):
        if gen_type is not None and gen.get("type") == gen_type:
            return idx
        if gen_type_key is not None and gen.get("gen_type") == gen_type_key:
            return idx
    raise RuntimeError("requested generator type not found")

if RUN_HARDWARE:
    SIGGEN_CH = find_first_gen(soccfg, gen_type="axis_signal_gen_v6")
    AWG_CH = find_first_gen(soccfg, gen_type_key="awg_tuning")
    RO_CH = 2 if len(soccfg["readouts"]) > 2 else 0
else:
    SIGGEN_CH = 0
    AWG_CH = 1
    RO_CH = 2

print("SIGGEN_CH:", SIGGEN_CH)
print("AWG_CH:", AWG_CH)
print("RO_CH:", RO_CH)


## Program using the FIR DDR sample-capture path

The program produces a normal signal-generator pulse, emits AWG tuning SET/RAMP commands, starts a readout window, and pulses the DDR4 trigger. The DRAM IP is armed outside the program with `soc.arm_ddr4_fir_samples()` before the tProcessor program starts.


In [ ]:
class QstlAwgTuningFirDdrSampleProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg
        self.siggen_ch = cfg["siggen_ch"]
        self.awg_ch = cfg["awg_ch"]
        self.ro_ch = cfg["ro_ch"]

        self.declare_gen(
            ch=self.siggen_ch,
            nqz=1,
        )
        self.declare_readout(
            ch=self.ro_ch,
            length=cfg["readout_length"],
        )

        self.set_pulse_registers(
            ch=self.siggen_ch,
            style="const",
            freq=cfg["siggen_freq"],
            phase=0,
            gain=cfg["siggen_gain"],
            length=cfg["siggen_length"],
            phrst=1,
            stdysel="zero",
        )
        self.set_readout_registers(
            ch=self.ro_ch,
            freq=cfg["readout_freq"],
            length=cfg["readout_length"],
            phrst=0,
        )

        self.synci(100)

    def body(self):
        cfg = self.cfg

        self.pulse(
            ch=self.siggen_ch,
            t=cfg["siggen_t"],
        )

        self.awg_set(
            ch=self.awg_ch,
            value=cfg["awg_start"],
            duration=cfg["awg_set_duration"],
            t=cfg["awg_set_t"],
        )
        self.awg_ramp(
            ch=self.awg_ch,
            target=cfg["awg_target"],
            duration=cfg["awg_ramp_duration"],
            t="auto",
        )

        self.readout(
            ch=self.ro_ch,
            t=cfg["readout_t"],
        )
        self.trigger(
            adcs=[self.ro_ch],
            ddr4=True,
            adc_trig_offset=cfg["adc_trig_offset"],
            t=cfg["trigger_t"],
            width=cfg["trigger_width"],
        )

        self.sync_all(cfg["relax_delay_cycles"])



## Configure the program

`ddr_samples_per_trigger` is the number of valid post-FIR 32-bit IQ words returned per DDR trigger. For the readout stream, one 32-bit word contains one signed int16 I sample and one signed int16 Q sample.

The pre-FIR readout window must be long enough to feed the 300-to-1 FIR and absorb filter group delay. The helper `soc.fir_readout_length_for_capture()` computes:

```text
readout_length >= ddr_samples_per_trigger * 300 + FIR_GROUP_DELAY_INPUT_SAMPLES + margin
```

The DDR sample buffer itself is armed with `sample_decim=1` through `soc.arm_ddr4_fir_samples()`.


In [ ]:
ddr_samples_per_trigger = 1000
readout_margin = 1024

prog_cfg = {
    "reps": 1,
    "siggen_ch": SIGGEN_CH,
    "awg_ch": AWG_CH,
    "ro_ch": RO_CH,
    "siggen_freq": 0,
    "siggen_gain": 30000,
    "siggen_length": 4096,
    "siggen_t": 80,
    "readout_freq": 0,
    "readout_length": None,
    "readout_t": 120,
    "trigger_t": 120,
    "adc_trig_offset": 0,
    "trigger_width": 12,
    "awg_start": 0,
    "awg_target": 2000,
    "awg_set_duration": 64,
    "awg_set_t": 40,
    "awg_ramp_duration": 256,
    "relax_delay_cycles": 500,
    "soft_avgs": 1,
}

ddr_triggers = prog_cfg["reps"]

if RUN_HARDWARE:
    prog_cfg["readout_length"] = soc.fir_readout_length_for_capture(
        ddr_samples_per_trigger,
        margin=readout_margin,
    )
    print("Requested post-FIR samples per trigger:", ddr_samples_per_trigger)
    print("Program readout length [pre-FIR samples]:", prog_cfg["readout_length"])
    print("FIR decimation:", soccfg["ddr4_buf"].get("fir_decimation"))
    print("FIR nominal output rate [MSPS]:", soccfg["ddr4_buf"].get("fir_output_fs_mhz"))
    prog = QstlAwgTuningFirDdrSampleProgram(soccfg, prog_cfg)
    print(prog)
else:
    prog_cfg["readout_length"] = ddr_samples_per_trigger * 300 + 8677 + readout_margin
    prog = None
    print("Program config prepared. Enable RUN_HARDWARE to instantiate with the board config.")


## Run FIR DDR capture

This arms the DDR sample buffer for post-FIR samples. Internally `arm_ddr4_fir_samples()` verifies that the FIR IP is present and programs the DDR sample IP with `sample_decim=1`.


In [ ]:
if RUN_HARDWARE:
    reserved_physical_words = soc.arm_ddr4_fir_samples(
        ch=RO_CH,
        n_samples=ddr_samples_per_trigger,
        n_triggers=ddr_triggers,
        address=0,
        stride_bytes=None,
        force_overwrite=False,
    )
    print("Reserved physical 32-bit words including zero padding:", reserved_physical_words)

    # acquire_decimated() runs the tProcessor program and also returns the normal decimated readout.
    decimated = prog.acquire_decimated(soc, progress=True)

    ddr_iq = soc.get_ddr4_fir_samples(
        n_samples=ddr_samples_per_trigger,
        n_triggers=ddr_triggers,
        start=0,
        stride_bytes=None,
    )
    print("DDR FIR IQ shape:", ddr_iq.shape)
else:
    ddr_iq = None
    print("Execution skipped. Set RUN_HARDWARE=True to arm DDR, run, and read data.")


## Plot captured DDR samples


In [ ]:
if RUN_HARDWARE and ddr_iq is not None:
    n_plot = min(len(ddr_iq), ddr_samples_per_trigger)
    t_us = np.arange(n_plot) / soccfg["ddr4_buf"].get("fir_output_fs_mhz", 1.0)

    plt.figure(figsize=(10, 4))
    plt.plot(t_us, ddr_iq[:n_plot, 0], label="I")
    plt.plot(t_us, ddr_iq[:n_plot, 1], label="Q")
    plt.xlabel("post-FIR time [us]")
    plt.ylabel("signed int16 sample")
    plt.title("qstl_awg_tuning_fir FIR DDR capture")
    plt.grid(True, alpha=0.3)
    plt.legend()
else:
    print("No DDR data to plot yet.")


## Optional multi-trigger readback shape

For multiple program repetitions, the DDR sample IP captures one event per trigger. `get_ddr4_samples()` returns valid samples with zero padding removed, so the data can be reshaped into `[trigger, sample, iq]` if every trigger uses the same sample count.


In [ ]:
if RUN_HARDWARE and ddr_iq is not None:
    ddr_events = ddr_iq.reshape((ddr_triggers, ddr_samples_per_trigger, 2))
    print("DDR FIR events shape:", ddr_events.shape)
else:
    print("Multi-trigger reshape example skipped.")
